# Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from scripts.isochrones.isochrones import build_isochrone_catchments
from scripts.tdlinx.data_loading import load_geodata
import polars as pl

GHSL_PATH = r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project_2\data\ghsl_raster_files\ghs_pop_germany_clipped.tif"


CACHE = Path(r"C:\edeka_cache")   # or wherever has space — local disk, not necessarily the ZEW share
CACHE.mkdir(parents=True, exist_ok=True)

# Data loading

In [ ]:
panel = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\panel")
gdf, kreis_gdf, bundesland_gdf = load_geodata()

In [ ]:
# nearest_competitors = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet")

# Calculate catchments for competitors

In [ ]:
from scripts.isochrones.network_voronoi_2 import prepare_stores, snap, load_cells, adapt_resolution, dedupe, candidates, travel_times, certification_report, save_matrix, assign, catchments as nv

In [ ]:
stores = snap(prepare_stores(panel))
cells  = load_cells(GHSL_PATH)
cells, m1 = adapt_resolution(cells, stores)     # 7M -> ~5M
cells_snapped  = pl.read_parquet("../outputs/catchments/cells_snapped3.parquet")
cells_snapped, m2 = dedupe(cells_snapped)                       # ~5M -> ~2M
cand   = candidates(cells_snapped, stores, k=20)
dur, _ = travel_times(cells_snapped, stores, cand, checkpoint="ckpt/")
certification_report(cand, dur, ranks=3)
save_matrix("matrix.npz", cells_snapped, stores, cand, dur)

a20 = assign(cells_snapped, stores, cand, dur, year=2024)
catchments(a2019)

In [ ]:
catchments(a2019)

In [ ]:
r1 = a2019.filter(pl.col("rank") == 1).join(
    cells.select("cell_id", "x", "y"), on="cell_id", how="left"
)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 12))
ax.scatter(r1["x"], r1["y"], c=r1["store_id"].hash().to_numpy(),
           s=1, cmap="tab20", linewidths=0)
ax.set_aspect("equal")

In [ ]:
cells_snapped = cells 
cells_snapped.write_parquet("../outputs/catchments/cells_snapped4.parquet")


In [ ]:
stores_snapped = stores
stores_snapped.write_parquet("../outputs/catchments/stores_snapped2.parquet")

In [ ]:
a2019.write_parquet("../outputs/catchments/attached2019_2.parquet")

In [ ]:
import numpy as np
def write_label_raster(
    cells: pl.DataFrame, assignment: pl.DataFrame, mapping: pl.DataFrame,
    template: str | Path, out_path: str | Path,
) -> None:
    """Burn rank-1 assignments back onto the original grid (int32, 0 = unassigned).
 
    `cells` is the *original* frame from load_cells (it carries row/col);
    `mapping` is compose(...) of every reduction applied. Codes are positional,
    with the store_id lookup written as a Parquet sidecar. Polygonise from this
    only if you genuinely need vectors -- the raster is cheaper for area and
    population arithmetic.
    """
    import rasterio
 
    r1 = assignment.filter(pl.col("rank") == 1)
    lookup = (pl.DataFrame({"store_id": r1["store_id"].unique(maintain_order=True)})
                .with_row_index("code", offset=1))
    burn = (mapping.join(r1.select(pl.col("cell_id").alias("dst"), "store_id"),
                         on="dst", how="inner")
                   .join(lookup, on="store_id", how="left")
                   .join(cells.select(pl.col("cell_id").alias("src"), "row", "col"),
                         on="src", how="inner"))
 
    with rasterio.open(template) as src:
        profile, shape = src.profile, (src.height, src.width)
    profile.update(dtype="int32", count=1, nodata=0, compress="lzw")
 
    grid = np.zeros(shape, np.int32)
    grid[burn["row"].to_numpy(), burn["col"].to_numpy()] = burn["code"].to_numpy()
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(grid, 1)
    lookup.write_parquet(Path(out_path).with_suffix(".lookup.parquet"))


In [ ]:
def compose(*maps: pl.DataFrame) -> pl.DataFrame:
    """Chain (src, dst) mappings so original cell_id -> final solve node."""
    m = maps[0]
    for nxt in maps[1:]:
        m = (m.join(nxt.rename({"src": "dst", "dst": "_d2"}), on="dst", how="left")
               .select("src", pl.col("_d2").alias("dst")))
    return m


In [ ]:
cells_orig = load_cells(GHSL_PATH)
cells_adapted, m1 = adapt_resolution(cells_orig, stores_snapped)



In [ ]:
write_label_raster(
    cells=cells_orig,        # <- original, not c2
    assignment=a2019,
    mapping=mapping,
    template=GHSL_PATH,
    out_path=("../outputs/catchments/catchments_2019_2.tif")
)

In [ ]:
m1.write_parquet("../outputs/catchments/map_adapt_2.parquet")
m2.write_parquet("../outputs/catchments/map_dedupe_2.parquet")
mapping.write_parquet("../outputs/catchments/map_full_2.parquet")   # compose(m1, m2)

In [ ]:
def plot_catchments(
    assignment: pl.DataFrame,
    cells: pl.DataFrame,
    stores: pl.DataFrame,
    *,
    by: str = "Gruppe",
    ax=None,
    s: float = 0.4,
    title: str = "",
    top: int = 12,
    boundaries: str | Path | None = None,
):
    """Scatter every solve node coloured by the Gruppe of its rank-1 store.
 
    ``cells`` must be the frame the assignment is keyed on (post-dedupe), since
    it supplies x/y. Groups outside the top-N by node count are collapsed to
    'other' -- a legend with 40 chains is unreadable and the tail is invisible on
    the map anyway.
 
    Drawn as a dense scatter rather than polygons: at millions of nodes this is
    what the computation actually produced, and it shows genuine ragged catchment
    edges instead of implying clean boundaries that were never estimated.
    """
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
 
    r1 = (assignment.filter(pl.col("rank") == 1)
          .join(stores.select("store_id", by), on="store_id", how="left")
          .join(cells.select("cell_id", "x", "y"), on="cell_id", how="inner")
          .with_columns(pl.col(by).fill_null("UNKNOWN")))
 
    keep = (r1.group_by(by).agg(pl.len().alias("n"))
              .sort("n", descending=True).head(top)[by].to_list())
    r1 = r1.with_columns(
        pl.when(pl.col(by).is_in(keep)).then(pl.col(by)).otherwise(pl.lit("other")).alias("_g")
    )
    cats = keep + (["other"] if (r1["_g"] == "other").any() else [])
    cmap = plt.get_cmap("tab20")
    colours = {g: cmap(i % 20) for i, g in enumerate(cats)}
 
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 11))
    codes = r1["_g"].to_list()
    ax.scatter(r1["x"].to_numpy(), r1["y"].to_numpy(),
               c=[colours[g] for g in codes], s=s, linewidths=0, rasterized=True)
    if boundaries is not None:
        import geopandas as gpd
        gpd.read_file(boundaries).to_crs(3035).boundary.plot(
            ax=ax, color="black", linewidth=0.3, alpha=0.4)
    ax.set_aspect("equal")
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=colours[g], label=g)
                       for g in cats], loc="upper left", fontsize=7, framealpha=0.9)
    return ax


In [ ]:
plot_catchments(
    assignment = a2019,
    cells = cells_snapped,
    stores = stores_snapped,
    by = "Gruppe",
    ax=None,
    s = 0.4,
    title = "",
    top = 12,
    boundaries = None,
)

In [ ]:
a2019

# Old implementation

In [ ]:
catchments = build_isochrone_catchments(
    panel, year=2024, clip_to=gdf,
    # shop_type_groups=None,
    shop_type=["Supermarkt", "Discount Geschäft"],
    resolution_m=1000,
    k_candidates=20,
    host="http://localhost:5000",
    # checkpoint_path="isochrone_grocery_2024.npz",
)

catchments.to_file("isochrone_grocery_2024_candidates20_resolution_1000.gpkg", driver="GPKG")

In [ ]:
import geopandas as gpd
catchments = gpd.read_file("isochrone_grocery_2024_candidates20_resolution_1000.gpkg")

In [ ]:
from scripts.visualizations.plot_catchments import plot_catchments, plot_catchments_grid

# Single region — works for either build_voronoi() or build_isochrone_catchments() output
plot_catchments("Berlin", catchments, kreis_gdf=kreis_gdf)
plot_catchments("Baden-Württemberg", catchments, bundesland_gdf=bundesland_gdf)

# Compare several Kreise side by side
plot_catchments_grid(
    ["Mannheim", "Heidelberg", "Karlsruhe", "Rhein-Neckar-Kreis"],
    catchments, kreis_gdf=kreis_gdf, ncols=2,
)

In [ ]:
from scripts.isochrones.route_nearest_competitors import route_nearest_competitors, attach_road_distances, consolidate_parts

route_nearest_competitors(
    pairs_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet",
    out_dir=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances_parts",
    host="http://192.168.70.123:5000",
)
consolidate_parts(r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances_parts", r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances.parquet")

In [ ]:
enriched = attach_road_distances(
    pairs_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors.parquet",
    road_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\road_distances.parquet",
    out_path=r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data\nearest competitors\nearest_competitors_with_road.parquet",
)